# Feature Engineering: Authors Dataset (Pandas)

This notebook performs data quality checks and feature engineering on the `authors.parquet` dataset using **Pandas**.

## Objectives
1. Assess data quality (null values, format issues)
2. Extract `birth_year` from `birth_date` (various formats)
3. Clean `name` field
4. Remove rows with no meaningful data
5. Generate quality report

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Configuration
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

## Step 1: Load and Inspect Raw Data

In [3]:
# Load authors data
input_path = PROCESSED_DIR / 'authors.parquet'
df = pd.read_parquet(input_path)

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSchema:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Total rows: 14,970,140
Columns: ['author_key', 'name', 'birth_date']

Schema:
author_key    str
name          str
birth_date    str
dtype: object

Memory usage: 915.48 MB


In [6]:
# Display first few rows
print(df.head(10))
print(df.tail(10))


             author_key                       name birth_date
0  /authors/OL10000133A                 Tibor Rácz        NaN
1   /authors/OL1000029A                  Farābī.        NaN
2  /authors/OL10000517A  Vanessa-Daneen Düsseldorf        NaN
3  /authors/OL10000576A                 Iris Woods        NaN
4  /authors/OL10000727A            Stefan Schakeit        NaN
5  /authors/OL10000765A             Károly Lencsés        NaN
6  /authors/OL10001024A              Dr. H. A&apos        NaN
7  /authors/OL10001866A            Lorna E. Walker        NaN
8  /authors/OL10001909A                 Debra Tate        NaN
9  /authors/OL10002212A               Ruben Llinas        NaN
                   author_key                    name birth_date
14970130  /authors/OL9998495A      Diana Dumetz Carry        NaN
14970131  /authors/OL9998570A         Ronald J. Nuzzi        NaN
14970132  /authors/OL9998748A               Von Bulow        NaN
14970133   /authors/OL999874A  Sayed Hamid A. Hurreiz     

## Step 2: Data Quality Assessment

In [4]:
# Check null values
print("=== Null Value Counts ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

quality_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null Count': null_counts.values,
    'Null Percentage': null_pct.values
})
print(quality_df.to_string(index=False))

=== Null Value Counts ===
    Column  Null Count  Null Percentage
author_key           0         0.000000
      name        2314         0.015457
birth_date    13055681        87.211482


In [11]:
# Sample birth_date values to understand format patterns
print("=== Sample birth_date Values ===")
sample_dates = df['birth_date'].dropna().head(50)
for i, date_val in enumerate(sample_dates,1):
    print(f"{i:2d}. '{date_val}'")

=== Sample birth_date Values ===
 1. '1849'
 2. '1951'
 3. '1912'
 4. '1945'
 5. '1899'
 6. '1929'
 7. '1933'
 8. '1798'
 9. '1952'
10. '1963'
11. '1857'
12. '1944'
13. '1892'
14. '1936'
15. 'fl. 1587'
16. '1928'
17. '1860'
18. '1967'
19. '1932'
20. '1928'
21. '1960'
22. '1909'
23. '1859'
24. '1938'
25. '1938'
26. '1957'
27. '1927'
28. '1940'
29. '1960'
30. '1923'
31. '1963'
32. '1956'
33. '1856'
34. '1945'
35. '1907'
36. '1957'
37. '1937 Apr. 1'
38. '1908'
39. '1948'
40. '1921'
41. '1942'
42. '1949'
43. '1942'
44. '1969'
45. '1958'
46. '1932'
47. '1908'
48. '1930'
49. '1919'
50. '(1977'


In [22]:
# Analyze birth_date format patterns
print("=== Birth Date Format Analysis ===")
non_null_dates = df['birth_date'].dropna()

if len(non_null_dates) > 0:
    # Count different patterns
    patterns = {
        '4-digit year only': non_null_dates.str.match(r'^\d{4}$', na=False).sum(),
        'ISO date (YYYY-MM-DD)': non_null_dates.str.match(r'^\d{4}-\d{2}-\d{2}$', na=False).sum(),
        'Partial date (YYYY-MM)': non_null_dates.str.match(r'^\d{4}-\d{2}$', na=False).sum(),
        'Contains "c." or "ca."': non_null_dates.str.contains(r'c\.|ca\.', case=False, na=False).sum(),
        'Contains range (YYYY-YYYY)': non_null_dates.str.contains(r'\d{4}\s*-\s*\d{4}', na=False).sum(),
        # Additional patterns to capture remaining birth dates
        'fl. (floruit)': non_null_dates.str.match(r'^fl\.?\s*\d{4}', na=False).sum(),
        'b. (born)': non_null_dates.str.match(r'^b\.?\s*\d{4}', na=False).sum(),
        'd. (died)': non_null_dates.str.match(r'^d\.?\s*\d{4}', na=False).sum(),
        'Year then month/day (e.g. 1937 Apr. 1)': non_null_dates.str.match(r'^\d{4}\s+[A-Za-z]', na=False).sum(),
        'Year in parentheses or brackets': non_null_dates.str.match(r'^[\\(\[]\s*\d{4}', na=False).sum(),
        "Decade (1850s or 1850's)": non_null_dates.str.contains(r"\d{4}'?s\b", na=False).sum(),
        'circa / about / ~': non_null_dates.str.contains(r'circa|about|~', case=False, na=False).sum(),
        'est. / estimated': non_null_dates.str.contains(r'est\.?|estimated', case=False, na=False).sum(),
        "Year with trailing ? or .": non_null_dates.str.match(r"^\d{4}[\.\?]\s*$", na=False).sum(),
        'Slash or dot date': non_null_dates.str.contains(r'\d{4}[/\\.]\d{1,2}[/\\.]\d{1,2}|\d{1,2}[/\\.]\d{1,2}[/\\.]\d{4}', na=False).sum(),
        'Month name + year': non_null_dates.str.contains(r'(January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\\.]?\s+\d{4}', case=False, na=False).sum(),
    }
    
    for pattern, count in patterns.items():
        pct = (count / len(non_null_dates)) * 100
        print(f"{pattern}: {count:,} ({pct:.2f}%)")
    
    has_four_digit_year = non_null_dates.str.contains(r'\d{4}', na=False).sum()
    print(f"\nContains any 4-digit year: {has_four_digit_year:,} ({has_four_digit_year/len(non_null_dates)*100:.2f}%)")

=== Birth Date Format Analysis ===
4-digit year only: 1,822,705 (95.21%)
ISO date (YYYY-MM-DD): 2,239 (0.12%)
Partial date (YYYY-MM): 21 (0.00%)
Contains "c." or "ca.": 5,799 (0.30%)
Contains range (YYYY-YYYY): 28 (0.00%)
fl. (floruit): 2,472 (0.13%)
b. (born): 213 (0.01%)
d. (died): 104 (0.01%)
Year then month/day (e.g. 1937 Apr. 1): 18,681 (0.98%)
Year in parentheses or brackets: 8,896 (0.46%)
Decade (1850s or 1850's): 10 (0.00%)
circa / about / ~: 84 (0.00%)
est. / estimated: 7 (0.00%)
Year with trailing ? or .: 10,171 (0.53%)
Slash or dot date: 849 (0.04%)
Month name + year: 15,015 (0.78%)

Contains any 4-digit year: 1,899,098 (99.20%)


/var/folders/9c/878pcq3j5vs8nslt0z5sj8380000gn/T/ipykernel_26109/554764682.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  'Month name + year': non_null_dates.str.contains(r'(January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\\.]?\s+\d{4}', case=False, na=False).sum(),


## Step 3: Extract Birth Year from Various Formats

In [25]:
def extract_year_from_date(date_str):
    """
    Extract year from various date formats found in Open Library data.
    
    Handles formats like:
    - "1850" (4-digit year)
    - "1850-01-15" (ISO date)
    - "1850-01" (partial date)
    - "January 1850" (text date)
    - "c. 1850" or "ca. 1850" (approximate)
    - "1850-1900" (range - extracts first year)
    - "1850?" (uncertain)
    
    Returns:
        Integer year if extractable, None otherwise
    """
    if pd.isna(date_str) or date_str == "":
        return None
    
    date_str = str(date_str).strip()
    
    # Pattern 1: Simple 4-digit year at start (with optional trailing ? or .)
    match = re.match(r'^(\d{4})[\.\?]?\s*$', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 2: ISO date format (YYYY-MM-DD or YYYY-MM)
    match = re.match(r'^(\d{4})-\d{1,2}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 3: Approximate (c., ca., circa, about, ~, est.)
    match = re.search(r'c\.?\s*(\d{4})|ca\.?\s*(\d{4})|circa\s*(\d{4})|about\s*(\d{4})|~\s*(\d{4})|est\.?\s*(\d{4})|estimated\s*(\d{4})', date_str, re.IGNORECASE)
    if match:
        year = int(next(g for g in match.groups() if g is not None))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 4: fl. (floruit), b. (born), d. (died)
    match = re.match(r'^(?:fl|b|d)\.?\s*(\d{4})', date_str, re.IGNORECASE)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 5: Year at start then space and month (e.g. 1937 Apr. 1)
    match = re.match(r'^(\d{4})\s+', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 6: Year in parentheses or brackets: (1850 or [1850
    match = re.match(r'^[\\(\[]\s*(\d{4})[\\)\]]?\s*', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 7: Range (1850-1900)
    match = re.match(r'(\d{4})\s*-\s*\d{4}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 8: Decade (1850s or 1850's)
    match = re.match(r"^(\d{4})'?s\b", date_str, re.IGNORECASE)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 9: Slash or dot dates (YYYY/MM/DD or DD/MM/YYYY)
    match = re.match(r'^(\d{4})[/\.]\d{1,2}[/\.]\d{1,2}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    match = re.search(r'(\d{4})[/\.]\d{1,2}[/\.]\d{1,2}', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 10: Month name + year (January 1850, Jan 1850)
    match = re.search(r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\.]?\s+(\d{4})\b', date_str, re.IGNORECASE)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 11: Any 4-digit number with word boundary (fallback)
    match = re.search(r'\b(\d{4})\b', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    # Pattern 12: Any 4-digit number (no word boundary, for special chars)
    match = re.search(r'(\d{4})', date_str)
    if match:
        year = int(match.group(1))
        if 0 <= year <= 2026:
            return year
    
    return None

In [27]:
# Test the extract_year_from_date function on sample values
test_dates = [
    "1850",
    "1850-01-15",
    "1850-01",
    "January 1850",
    "c. 1850",
    "ca. 1850",
    "1850-1900",
    "1850?",
    "fl. 1587",
    "b. 1920",
    "1937 Apr. 1",
    "(1977",
    "[1850]",
    "1850s",
    "circa 1850",
    "est. 1900",
    "1850.",
    None,
    "",
    "invalid"
]

print("Testing year extraction:")
for date_str in test_dates:
    year = extract_year_from_date(date_str)
    print(f"  '{date_str}' -> {year}")

Testing year extraction:
  '1850' -> 1850
  '1850-01-15' -> 1850
  '1850-01' -> 1850
  'January 1850' -> 1850
  'c. 1850' -> 1850
  'ca. 1850' -> 1850
  '1850-1900' -> 1850
  '1850?' -> 1850
  'fl. 1587' -> 1587
  'b. 1920' -> 1920
  '1937 Apr. 1' -> 1937
  '(1977' -> 1977
  '[1850]' -> 1850
  '1850s' -> 1850
  'circa 1850' -> 1850
  'est. 1900' -> 1900
  '1850.' -> 1850
  'None' -> None
  '' -> None
  'invalid' -> None


## Step 4: Clean and Transform Data

In [28]:
# Create a copy for cleaning
df_cleaned = df.copy()

print(f"Original row count: {len(df_cleaned):,}")

Original row count: 14,970,140


In [29]:
# Extract birth_year from birth_date
print("Extracting birth_year from birth_date...")
df_cleaned['birth_year'] = df_cleaned['birth_date'].apply(extract_year_from_date)

# Convert to Int16 (nullable integer)
df_cleaned['birth_year'] = df_cleaned['birth_year'].astype('Int16')

print(f"Rows with valid birth_year: {df_cleaned['birth_year'].notna().sum():,}")
print(f"Rows with null birth_year: {df_cleaned['birth_year'].isna().sum():,}")

Extracting birth_year from birth_date...
Rows with valid birth_year: 1,899,078
Rows with null birth_year: 13,071,062


In [30]:
# Clean name field (trim whitespace)
print("Cleaning name field...")
df_cleaned['name'] = df_cleaned['name'].str.strip()

# Remove rows with no meaningful data
# Keep rows that have at least author_key and name
print("\nFiltering rows...")
before_filter = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['author_key'].notna() & 
    df_cleaned['name'].notna() & 
    (df_cleaned['name'] != "")
].copy()

rows_removed = before_filter - len(df_cleaned)
print(f"Rows removed: {rows_removed:,}")
print(f"Rows retained: {len(df_cleaned):,}")

Cleaning name field...

Filtering rows...
Rows removed: 2,315
Rows retained: 14,967,825


In [31]:
# Select final columns (drop birth_date, keep birth_year)
df_cleaned = df_cleaned[["author_key", "name", "birth_year"]].copy()

print("Final schema:")
print(df_cleaned.dtypes)
print(f"\nFinal row count: {len(df_cleaned):,}")

Final schema:
author_key      str
name            str
birth_year    Int16
dtype: object

Final row count: 14,967,825


## Step 5: Final Quality Check

In [32]:
# Final quality check
print("=== Final Quality Check ===")
valid_birth_year = df_cleaned['birth_year'].notna().sum()
pct_valid = (valid_birth_year / len(df_cleaned)) * 100

print(f"Rows with valid birth_year: {valid_birth_year:,} ({pct_valid:.2f}%)")
print(f"Rows with null birth_year: {len(df_cleaned) - valid_birth_year:,} ({(100-pct_valid):.2f}%)")

=== Final Quality Check ===
Rows with valid birth_year: 1,899,075 (12.69%)
Rows with null birth_year: 13,068,750 (87.31%)


In [33]:
# Birth year statistics
if valid_birth_year > 0:
    year_stats = df_cleaned['birth_year'].describe()
    print("\n=== Birth Year Statistics ===")
    print(f"Minimum year: {df_cleaned['birth_year'].min()}")
    print(f"Maximum year: {df_cleaned['birth_year'].max()}")
    print(f"Median year: {df_cleaned['birth_year'].median():.0f}")
    print(f"Mean year: {df_cleaned['birth_year'].mean():.0f}")
    print(f"\nDetailed statistics:")
    print(year_stats)


=== Birth Year Statistics ===
Minimum year: 4
Maximum year: 2025
Median year: 1928
Mean year: 1904

Detailed statistics:
count      1899075.0
mean     1903.751739
std        85.615215
min              4.0
25%           1884.0
50%           1928.0
75%           1951.0
max           2025.0
Name: birth_year, dtype: Float64


In [34]:
# Display sample of cleaned data
print("\n=== Sample of Cleaned Data ===")
df_cleaned.head(20)


=== Sample of Cleaned Data ===


,author_key,name,birth_year
0,/authors/OL10000133A,Tibor Rácz,<NA>
1,/authors/OL1000029A,Farābī.,<NA>
2,/authors/OL10000517A,Vanessa-Daneen Düsseldorf,<NA>
3,/authors/OL10000576A,Iris Woods,<NA>
4,/authors/OL10000727A,Stefan Schakeit,<NA>
5,/authors/OL10000765A,Károly Lencsés,<NA>
6,/authors/OL10001024A,Dr. H. A&apos,<NA>
7,/authors/OL10001866A,Lorna E. Walker,<NA>
8,/authors/OL10001909A,Debra Tate,<NA>
9,/authors/OL10002212A,Ruben Llinas,<NA>


## Step 6: Save Cleaned Data

In [40]:
# Save cleaned data
output_path = PROCESSED_DIR / 'authors_cleaned.parquet'
df_cleaned.to_parquet(output_path, index=False)
print(f"✓ Saved cleaned data to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

✓ Saved cleaned data to ../data/processed/authors_cleaned.parquet
File size: 366.18 MB


## Step 7: Generate Quality Report

In [41]:
# Generate quality report
report_path = REPORTS_DIR / 'data_quality_authors_pandas.md'

valid_birth_year_count = df_cleaned['birth_year'].notna().sum()
null_birth_year_count = df_cleaned['birth_year'].isna().sum()

report = f"""# Data Quality Report: Authors Dataset (Pandas)

## Summary
- **Original row count**: {len(df):,}
- **Cleaned row count**: {len(df_cleaned):,}
- **Rows removed**: {len(df) - len(df_cleaned):,} ({(len(df) - len(df_cleaned))/len(df)*100:.2f}%)
- **Rows retained**: {len(df_cleaned)/len(df)*100:.2f}%

## Data Quality Metrics

### Birth Year Extraction
- **Rows with valid birth_year**: {valid_birth_year_count:,} ({valid_birth_year_count/len(df_cleaned)*100:.2f}%)
- **Rows with null birth_year**: {null_birth_year_count:,} ({null_birth_year_count/len(df_cleaned)*100:.2f}%)

"""

if valid_birth_year_count > 0:
    report += f"""### Birth Year Statistics
- **Minimum year**: {df_cleaned['birth_year'].min()}
- **Maximum year**: {df_cleaned['birth_year'].max()}
- **Median year**: {df_cleaned['birth_year'].median():.0f}
- **Mean year**: {df_cleaned['birth_year'].mean():.0f}

"""

report += f"""## Schema Changes
- **Removed**: `birth_date` (string, various formats)
- **Added**: `birth_year` (Int16, standardized year)

## Cleaning Steps Applied
1. Extracted year from `birth_date` using regex patterns
2. Validated years (range: 0-2026)
3. Trimmed whitespace from `name` field
4. Removed rows with null/empty `author_key` or `name`

## Tools Used
- **Library**: Pandas
- **Date Extraction**: Custom regex-based function
- **Output Format**: Parquet (columnar, compressed)
"""

report_path.write_text(report)
print(f"✓ Quality report saved to {report_path}")

✓ Quality report saved to ../reports/data_quality_authors_pandas.md


In [39]:
# Display the report
print("\n=== Quality Report ===")
print(report_path.read_text())


=== Quality Report ===
# Data Quality Report: Authors Dataset (Pandas)

## Summary
- **Original row count**: 14,970,140
- **Cleaned row count**: 14,967,825
- **Rows removed**: 2,315 (0.02%)
- **Rows retained**: 99.98%

## Data Quality Metrics

### Birth Year Extraction
- **Rows with valid birth_year**: 1,899,075 (12.69%)
- **Rows with null birth_year**: 13,068,750 (87.31%)

### Birth Year Statistics
- **Minimum year**: 4
- **Maximum year**: 2025
- **Median year**: 1928
- **Mean year**: 1904

## Schema Changes
- **Removed**: `birth_date` (string, various formats)
- **Added**: `birth_year` (Int16, standardized year)

## Cleaning Steps Applied
1. Extracted year from `birth_date` using regex patterns
2. Validated years (range: 0-2026)
3. Trimmed whitespace from `name` field
4. Removed rows with null/empty `author_key` or `name`

## Tools Used
- **Library**: Pandas
- **Date Extraction**: Custom regex-based function
- **Output Format**: Parquet (columnar, compressed)



## Summary

✓ Data loaded and inspected
✓ Data quality assessed
✓ Birth year extracted from various date formats
✓ Name field cleaned
✓ Invalid rows removed
✓ Cleaned data saved to `authors_cleaned.parquet`
✓ Quality report generated

**Next Steps**: Use `authors_cleaned.parquet` for further analysis or proceed with other datasets.